In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 8 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251211_164402.csv
Loaded: NBA_DFS_20251211_164531.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Anfernee Simons,Over,11.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
1,Underdog,player_points,Anfernee Simons,Under,11.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
2,Underdog,player_points,Jaylen Brown,Over,29.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
3,Underdog,player_points,Jaylen Brown,Under,29.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
4,Underdog,player_points,Myles Turner,Over,12.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## For Post Analysis

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

singleBets = calculateSingleBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,                  
    max_player_appearances=1,  
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)
singleBets.to_csv(f'notebooks/exploration/old_evs/singleBets_UD_{current_date}.csv', index=False)
singleBets

Computing predictions for 94 players...
[MIN] No data found for Ron Holland
Found 93 valid players


,NAME,LINE,SIDE,PREDICTION,MODEL_PROB,IMPLIED_PROB,EDGE,BET_EDGE,ODDS,DECIMAL_ODDS,EV,EV_PERCENT,KELLY_QUARTER,TEAM,OPPONENT
66,T.J. McConnell,6.5,over,14.91,0.975,0.56,8.41,0.415,-137,1.730,0.6863,68.63,0.2351,IND,PHI
10,Jabari Smith Jr.,14.5,over,20.02,0.831,0.49,5.52,0.341,100,2.000,0.6615,66.15,0.1654,HOU,LAC
12,Amen Thompson,16.5,over,22.75,0.866,0.51,6.25,0.356,-111,1.901,0.6465,64.65,0.1794,HOU,LAC
0,Anfernee Simons,11.5,over,16.62,0.829,0.50,5.12,0.329,-103,1.971,0.6330,63.30,0.1630,BOS,MIL
33,Maxime Raynaud,11.5,under,5.31,0.830,0.50,6.19,0.330,-106,1.943,0.6128,61.28,0.1624,SAC,DEN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Dominick Barlow,6.5,under,5.42,0.461,0.56,1.08,-0.099,-137,1.730,-0.2026,-20.26,0.0000,PHI,IND
59,Pascal Siakam,23.5,under,22.66,0.458,0.56,0.84,-0.102,-137,1.730,-0.2085,-20.85,0.0000,IND,PHI
41,Luke Kennard,6.5,under,6.24,0.404,0.56,0.26,-0.156,-137,1.730,-0.3019,-30.19,0.0000,ATL,DET
13,Nicolas Batum,4.5,under,4.17,0.373,0.53,0.33,-0.157,-119,1.840,-0.3136,-31.36,0.0000,LAC,HOU


In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=100,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)


prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/prizepicksPairs_{current_date}.csv', index=False)
# prizepicksPairs.to_csv(f'notebooks/exploration/old_evs/underdogPairs_{current_date}_{today}.csv', index=False)
prizepicksPairs

Computing predictions for 129 players...
[MIN] No data found for Herb Jones
[MIN] No data found for Wendell Carter Jr
Found 127 valid players
Generated 7259 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,IMPLIED_PROB 1,IMPLIED_PROB 2,PARLAY_PROB,PARLAY_IMPLIED_PROB,PARLAY_EDGE,EDGE 1,EDGE 2,ODDS 1,ODDS 2,PARLAY_ODDS,PARLAY_DECIMAL,EV,EV_PERCENT,KELLY_QUARTER
3862,Jordan Poole,Justin Champagnie,12.5,9.5,over,under,20.83,1.64,0.926,0.954,0.52,0.53,0.883,0.276,0.608,8.33,7.86,-116,-118,244,3.44,2.0391,203.91,0.2089
900,Anfernee Simons,Amen Thompson,11.5,16.5,over,over,16.62,22.75,0.829,0.866,0.50,0.51,0.718,0.255,0.463,5.12,6.25,-103,-111,275,3.75,1.6912,169.12,0.1537
2477,Reed Sheppard,Maxime Raynaud,10.5,11.5,over,under,16.25,5.31,0.860,0.830,0.52,0.50,0.714,0.260,0.454,5.75,6.19,-115,-106,263,3.63,1.5906,159.06,0.1512
1495,Josh Minott,Nique Clifford,6.5,8.5,over,under,9.19,2.67,0.739,0.829,0.48,0.54,0.612,0.259,0.353,2.69,5.83,104,-123,270,3.70,1.2657,126.57,0.1172
2277,Jabari Smith Jr.,Russell Westbrook,14.0,17.5,over,under,20.02,10.85,0.854,0.792,0.56,0.52,0.676,0.291,0.385,6.02,6.65,-137,-115,223,3.23,1.1835,118.35,0.1327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5111,Precious Achiuwa,Rudy Gobert,7.5,9.5,under,under,7.01,8.81,0.412,0.429,0.50,0.52,0.177,0.260,-0.083,0.49,0.69,-103,-112,273,3.73,-0.3412,-34.12,0.0000
5644,Ausar Thompson,Cason Wallace,11.5,7.5,under,under,10.92,6.82,0.424,0.420,0.52,0.51,0.178,0.265,-0.087,0.58,0.68,-112,-108,265,3.65,-0.3506,-35.06,0.0000
2719,Kris Dunn,Naz Reid,7.5,14.5,under,under,6.64,14.12,0.451,0.432,0.55,0.53,0.195,0.292,-0.097,0.86,0.38,-128,-119,228,3.28,-0.3615,-36.15,0.0000
750,Bobby Portis,Pascal Siakam,13.5,23.5,under,under,12.95,22.66,0.429,0.458,0.53,0.56,0.197,0.297,-0.100,0.55,0.84,-118,-137,220,3.20,-0.3712,-37.12,0.0000


## Top EVs for 2 leg bets

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 94 players...
[MIN] No data found for Ron Holland
Found 93 valid players
Generated 3868 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
887,Jabari Smith Jr.,T.J. McConnell,14.5,6.5,100,-137,20.02,14.91,0.831,0.975,over,over,0.810,246,180.20,0.1831
4,Anfernee Simons,Amen Thompson,11.5,16.5,-103,-111,16.62,22.75,0.829,0.866,over,over,0.718,275,169.12,0.1537
932,Reed Sheppard,Maxime Raynaud,10.5,11.5,-115,-106,16.25,5.31,0.860,0.830,over,under,0.714,263,159.06,0.1512
3322,KJ Simpson,Andre Drummond,12.5,4.5,-137,-137,2.99,10.16,0.929,0.910,under,over,0.845,199,152.74,0.1919
534,Josh Minott,Nique Clifford,6.5,8.5,104,-123,9.19,2.67,0.739,0.829,over,under,0.612,270,126.57,0.1172
2126,Russell Westbrook,Paolo Banchero,17.5,21.5,-115,-110,10.85,15.31,0.792,0.773,under,under,0.612,257,118.34,0.1151
1413,Shaedon Sharpe,Moses Moody,23.5,8.5,-115,-114,28.95,11.95,0.787,0.781,over,over,0.614,251,115.47,0.1150
2000,Jamal Murray,Stephen Curry,24.5,24.5,-106,-106,29.23,29.99,0.754,0.749,over,over,0.565,278,113.58,0.1021
293,Payton Pritchard,Brandon Miller,16.5,21.5,-110,-120,20.87,14.74,0.763,0.786,over,under,0.599,250,109.66,0.1097
115,Jaylen Brown,Tobias Harris,29.5,13.5,-102,-108,34.59,16.88,0.716,0.726,over,over,0.520,281,98.11,0.0873


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 129 players...
[MIN] No data found for Herb Jones
[MIN] No data found for Wendell Carter Jr
Found 127 valid players
Generated 7259 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
3862,Jordan Poole,Justin Champagnie,12.5,9.5,-116,-118,20.83,1.64,0.926,0.954,over,under,0.883,244,203.91,0.2089
900,Anfernee Simons,Amen Thompson,11.5,16.5,-103,-111,16.62,22.75,0.829,0.866,over,over,0.718,275,169.12,0.1537
2477,Reed Sheppard,Maxime Raynaud,10.5,11.5,-115,-106,16.25,5.31,0.860,0.830,over,under,0.714,263,159.06,0.1512
1495,Josh Minott,Nique Clifford,6.5,8.5,104,-123,9.19,2.67,0.739,0.829,over,under,0.612,270,126.57,0.1172
2277,Jabari Smith Jr.,Russell Westbrook,14.0,17.5,-137,-115,20.02,10.85,0.854,0.792,over,under,0.676,223,118.35,0.1327
3288,Shaedon Sharpe,Paolo Banchero,23.5,21.5,-115,-110,28.95,15.31,0.787,0.773,over,under,0.608,257,116.93,0.1137
4274,Jamal Murray,Moses Moody,24.5,8.5,-106,-114,29.23,11.95,0.754,0.781,over,over,0.589,265,114.93,0.1084
2910,Steven Adams,Stephen Curry,5.5,24.5,-106,-106,7.70,29.99,0.753,0.749,over,over,0.564,278,113.10,0.1017
556,Payton Pritchard,Devin Vassell,16.5,11.5,-110,-127,20.87,16.36,0.763,0.810,over,over,0.618,241,110.71,0.1148
1821,James Harden,Brandon Miller,22.5,21.5,-106,-120,26.50,14.74,0.742,0.786,over,under,0.583,256,107.62,0.1051


## 3 leg parlay

### Underdog picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 94 players...
[MIN] No data found for Ron Holland
Found 93 valid players
Generated 94754 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
207,Anfernee Simons,Jabari Smith Jr.,T.J. McConnell,11.5,14.5,6.5,-103,100,-137,16.62,20.02,14.91,0.829,0.831,0.975,over,over,over,0.671,582,357.62,0.1536
37236,Amen Thompson,Maxime Raynaud,KJ Simpson,16.5,11.5,12.5,-111,-106,-137,22.75,5.31,2.99,0.866,0.830,0.929,over,under,under,0.667,539,326.48,0.1514
19411,Josh Minott,Reed Sheppard,Andre Drummond,6.5,10.5,4.5,104,-115,-137,9.19,16.25,10.16,0.739,0.860,0.910,over,over,over,0.578,560,281.69,0.1258
48374,Shaedon Sharpe,Nique Clifford,Paolo Banchero,23.5,8.5,21.5,-115,-123,-110,28.95,2.67,15.31,0.787,0.829,0.773,over,under,under,0.504,547,225.90,0.1032
11005,Payton Pritchard,Russell Westbrook,Moses Moody,16.5,17.5,8.5,-110,-115,-114,20.87,10.85,11.95,0.763,0.792,0.781,over,under,over,0.471,570,215.67,0.0946


### Prizepicks picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 129 players...
[MIN] No data found for Herb Jones
[MIN] No data found for Wendell Carter Jr
Found 127 valid players
Generated 246251 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
100908,Amen Thompson,Jordan Poole,Justin Champagnie,16.5,12.5,9.5,-111,-116,-118,22.75,20.83,1.64,0.866,0.926,0.954,over,over,under,0.765,554,400.45,0.1807
45429,Anfernee Simons,Reed Sheppard,Maxime Raynaud,11.5,10.5,11.5,-103,-115,-106,16.62,16.25,5.31,0.829,0.860,0.830,over,over,under,0.591,616,323.38,0.1312
73133,Josh Minott,Jabari Smith Jr.,Nique Clifford,6.5,14.0,8.5,104,-137,-123,9.19,20.02,2.67,0.739,0.854,0.829,over,over,under,0.523,540,234.66,0.1086
148912,Shaedon Sharpe,Russell Westbrook,Paolo Banchero,23.5,17.5,21.5,-115,-115,-110,28.95,10.85,15.31,0.787,0.792,0.773,over,under,under,0.481,567,220.85,0.0974
134312,Steven Adams,Jamal Murray,Moses Moody,5.5,24.5,8.5,-106,-106,-114,7.70,29.23,11.95,0.753,0.754,0.781,over,over,over,0.443,609,214.27,0.0880
